# 05 - Model zona kedaluwarsa 4 KELAS

Melatih YOLOv8 untuk membedakan **`date` / `code` / `due` / `prod`**, menggantikan
model 1 kelas (`expired_date_zone`) yang terpasang sekarang.

**Kenapa:** jawaban salah `chees` berasal dari parser yang membaca **nomor produksi**
`210526` sebagai tanggal. Memperbaikinya dari sisi parser sudah dicoba 20-11 dan
**gagal** - teks OCR-nya tidak memuat label "Kode Produksi" sama sekali. Dengan model
4 kelas, wilayah kode produksi ditolak **sebelum** OCR, jadi labelnya tidak perlu terbaca.

**Harapan yang jujur:** batas atas perbaikannya sekitar **+6 poin** (1 berkas dari 16).
Jalur zona sudah terukur sebagai optimasi **latensi**, bukan akurasi - 81% sama dengan
maupun tanpa jalur zona (02-10). Jangan harapkan lompatan.

Aplikasi **sudah siap** menerima model ini: `pick_zone` memakai
`expiry.zone_classes_date` dan `zone_classes_reject`, dan mundur aman ke perilaku lama
selama kelasnya kosong. Jadi memasang bobot baru tidak butuh perubahan kode.


In [ ]:
# Sel 1 - PENJAGA GPU. Latihan k-fold uang v3 pernah diam-diam jalan di CPU
# (1 epoch = 965 detik, ~80 jam untuk 5 lipatan) karena !nvidia-smi hanya MENCETAK.
import torch
assert torch.cuda.is_available(), (
    'Runtime ini TANPA GPU. Runtime > Change runtime type > T4 GPU, lalu jalankan ulang.')
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
# Sel 2 - pustaka
%pip -q install ultralytics roboflow
import ultralytics
print('ultralytics', ultralytics.__version__)


In [ ]:
# Sel 3 - konfigurasi. ISI API KEY DI SINI (jangan di-commit).
RF_API_KEY   = ''          # dari app.roboflow.com > Settings > API Keys
WORKSPACE    = 'esigelec-rniez'
PROJECT      = 'expiry'
VERSI        = None        # None = versi terbaru

EPOCHS       = 80
IMGSZ        = 640         # samakan dengan vision.expiry.imgsz di settings.yaml
RUN_NAME     = 'zone_4kelas_v1'

# Kelas yang WAJIB ada. Kalau dataset ternyata lain, sel 5 berhenti keras -
# lebih baik gagal nyaring daripada melatih model yang kelasnya tidak cocok
# dengan config aplikasi.
KELAS_WAJIB  = {'date', 'code', 'due', 'prod'}
assert RF_API_KEY, 'Isi RF_API_KEY dulu.'


In [ ]:
# Sel 4 - unduh dataset
from roboflow import Roboflow
rf = Roboflow(api_key=RF_API_KEY)
proyek = rf.workspace(WORKSPACE).project(PROJECT)
versi = proyek.version(VERSI) if VERSI else proyek.versions()[0]
ds = versi.download('yolov8')
print('diunduh ke:', ds.location)


In [ ]:
# Sel 5 - PERIKSA DATASET SEBELUM MELATIH.
# Dua jebakan yang sudah pernah merusak latihan di proyek ini:
#   1. nama kelas tidak sesuai harapan (dataset Roboflow berlabel tertukar, 27-09)
#   2. gambar TANPA label di split train - dalam YOLO itu berarti 'tidak ada objek
#      di gambar ini', dan itu merusak k-fold uang pertama (07-10)
import collections
from pathlib import Path
import yaml

data_yaml = Path(ds.location) / 'data.yaml'
d = yaml.safe_load(data_yaml.read_text())
nama = {i: n for i, n in enumerate(d['names'])}
print('kelas:', nama)
kurang = KELAS_WAJIB - {n.lower() for n in nama.values()}
assert not kurang, f'Kelas yang diharapkan tidak ada: {kurang}. Periksa datasetnya.'

for split in ('train', 'valid', 'test'):
    img = Path(ds.location, split, 'images')
    lab = Path(ds.location, split, 'labels')
    if not img.is_dir():
        print(f'{split:6} TIDAK ADA')
        continue
    gambar = sorted(p for p in img.iterdir() if p.suffix.lower() in ('.jpg', '.jpeg', '.png'))
    kosong, hitung = 0, collections.Counter()
    for g in gambar:
        t = lab / (g.stem + '.txt')
        baris = [b for b in t.read_text().splitlines() if b.strip()] if t.exists() else []
        if not baris:
            kosong += 1
        for b in baris:
            hitung[nama.get(int(b.split()[0]), '?')] += 1
    print(f'{split:6} {len(gambar):5} gambar, {kosong:4} TANPA label, kotak: {dict(hitung)}')

print('Kalau jumlah TANPA label di train besar, HENTIKAN dan periksa dulu.')


In [ ]:
# Sel 6 - jangan menimpa hasil latihan yang sudah ada
hasil_dir = Path('runs/detect') / RUN_NAME
assert not (hasil_dir / 'weights' / 'best.pt').exists(), (
    f'{hasil_dir} sudah berisi bobot. Ganti RUN_NAME, atau hapus foldernya dengan sadar.')


In [ ]:
# Sel 7 - latih
from ultralytics import YOLO
model = YOLO('yolov8n.pt')
model.train(data=str(data_yaml), epochs=EPOCHS, imgsz=IMGSZ,
            name=RUN_NAME, project='runs/detect', seed=0, deterministic=True)


In [ ]:
# Sel 8 - evaluasi pada split TEST, per kelas
terbaik = YOLO(str(hasil_dir / 'weights' / 'best.pt'))
m = terbaik.val(data=str(data_yaml), split='test', imgsz=IMGSZ)
print(f'mAP50 {m.box.map50:.3f}  mAP50-95 {m.box.map:.3f}')
for i, n in nama.items():
    try:
        print(f'  {n:6} AP50 {m.box.ap50[i]:.3f}')
    except (IndexError, TypeError):
        pass
print('Yang paling menentukan untuk aplikasi: AP kelas `date` dan `code`.')
print('`code` penting karena gunanya MENOLAK, bukan membaca.')


In [ ]:
# Sel 9 - ukuran bobot = penanda latihan selesai normal
# Ultralytics membuang state optimizer HANYA saat latihan berakhir normal.
# ~6 MB = selesai; ~32 MB = terputus (pelajaran k-fold uang v3, 09-11).
b = hasil_dir / 'weights' / 'best.pt'
mb = b.stat().st_size / 1e6
print(f'{b} = {mb:.1f} MB ->', 'selesai normal' if mb < 15 else 'TERPUTUS, jangan dipakai')


## Memasang bobotnya

1. Unduh `runs/detect/zone_4kelas_v1/weights/best.pt`
2. Salin ke `models/best_zone_4kelas.pt` di repo (**jangan timpa** `best_zone.pt`,
   itu bukti angka 81% yang sekarang)
3. Di `config/settings.yaml` ubah `expiry.zone_weights` ke bobot baru
4. **Ukur sebelum menyimpulkan:**

```
python scripts\ukur_kedaluwarsa.py
```

Yang dibandingkan **benar / salah / diam**, bukan satu angka akurasi - arah kesalahan
di modul ini sudah tiga kali terbukti lebih penting daripada jumlahnya. Dasar
pembanding sekarang: **13 benar / 2 salah / 1 diam = 81%**.

Kalau `chees` berubah dari SALAH menjadi BENAR atau DIAM, perbaikannya nyata.
Kalau tidak ada yang berubah, bobot 4 kelas **tidak membeli apa pun** untuk data ini
dan itu harus dicatat apa adanya, bukan disimpan diam-diam.
